# Aula 14 — Validação cruzada na prática

**Aluno · prática guiada e laboratório**

Nesta prática, retomamos os 201 automóveis da aula de regressão. A pergunta é: **qual modelo tende a prever melhor o preço de carros novos?** Usaremos o conjunto de treino para comparar candidatos e guardaremos o teste até a decisão estar tomada.

**Ao final, você deverá:** explicar os cinco folds; usar `KFold` e `cross_val_score`; interpretar o sinal do MAE, a média e o desvio; comparar modelos nas mesmas partes; usar `Pipeline` quando houver transformação; reajustar o escolhido e escrever um relatório curto.

## Como trabalhar

Execute as células em ordem com **Shift + Enter**. Complete os blocos `TODO` e responda às perguntas em Markdown. Para trabalhar no Colab, envie também `automoveis_preparados.csv` (pasta `dados` ou diretório atual). Use apenas as variáveis de **treino** para comparar modelos. O teste será aberto na etapa final.

| Tempo | Etapa |
|---:|---|
| 0–25 min | Base, divisão e folds |
| 25–40 min | Exemplo guiado de validação cruzada |
| 40–60 min | Comparação de modelos |
| 60–75 min | Polinômios e `Pipeline` |
| 75–100 min | Escolha, teste e relatório |

## 1. Ambiente e dados

**Mesma base da aula de regressão:** `price` é o preço em dólares. Usaremos `horsepower` na regressão simples e `horsepower`, `curb-weight`, `engine-size`, `highway-mpg` na múltipla. O arquivo preparado já contém as decisões de limpeza dos encontros anteriores; nesta prática não ajustaremos transformações usando a base inteira.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

pd.set_option("display.float_format", lambda valor: f"{valor:,.2f}")
COLUNAS_MULTIPLAS = ["horsepower", "curb-weight", "engine-size", "highway-mpg"]

In [ ]:
caminhos = [
    Path("dados/automoveis_preparados.csv"),
    Path("aulas/014/dados/automoveis_preparados.csv"),
    Path("automoveis_preparados.csv"),
]
caminho_dados = next((p for p in caminhos if p.is_file()), None)
if caminho_dados is None:
    raise FileNotFoundError(
        "Coloque automoveis_preparados.csv em dados/ ou no diretório do notebook."
    )

df = pd.read_csv(caminho_dados)
colunas_necessarias = COLUNAS_MULTIPLAS + ["price"]
assert len(df) == 201, "A prática usa a base preparada com 201 carros."
assert df[colunas_necessarias].notna().all().all(), "Há valores ausentes nas variáveis da prática."
print(f"Arquivo: {caminho_dados} | carros: {len(df)}")
df[colunas_necessarias].head()

### Separe o teste antes de comparar

Uma única divisão 80/20 deixa **160 carros no treino** e **41 no teste**. As colunas da regressão simples serão selecionadas *dentro* do mesmo treino, para que os candidatos sejam avaliados nos mesmos carros. `random_state=42` torna a divisão reproduzível; não é um número mágico.

In [ ]:
X = df[COLUNAS_MULTIPLAS]
y = df["price"]
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.20, random_state=42
)
X_treino_s = X_treino[["horsepower"]]
X_treino_m = X_treino[COLUNAS_MULTIPLAS]
print(f"Treino: {len(X_treino)} carros | Teste guardado: {len(X_teste)} carros")
assert len(X_treino) == 160 and len(X_teste) == 41

> **Pare e explique:** por que não devemos escolher o modelo pelo MAE dos 41 carros de teste? Escreva uma frase antes de continuar.

**Sua resposta:** …

## 2. Como os 5 folds dividem o treino

`KFold` recebe somente os 160 carros de treino. Com `shuffle=True`, embaralha antes de cortar; a semente fixa permite reproduzir as mesmas partes. Em cada rodada, 128 carros ajustam o modelo e 32 validam. Cada carro passa pela validação uma vez.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
partes = []
indices_validacao = []
for numero, (i_ajuste, i_validacao) in enumerate(kf.split(X_treino), start=1):
    partes.append({"parte": numero, "ajuste": len(i_ajuste), "validacao": len(i_validacao)})
    indices_validacao.extend(i_validacao)

assert len(indices_validacao) == len(set(indices_validacao)) == len(X_treino)
pd.DataFrame(partes).set_index("parte")

> **Pare e explique:** quantas vezes um carro do treino aparece na validação? Quantos carros ajustam o modelo em cada rodada? Por que usamos `shuffle=True` nesta base, que está ordenada por marca?

**Sua resposta:** …

## 3. Exemplo guiado: regressão múltipla

O scikit-learn trata toda pontuação como “maior é melhor”. Por isso `scoring="neg_mean_absolute_error"` devolve o **MAE com sinal negativo**. Multiplicamos por −1 para voltar a valores de erro em US$; daí **menor MAE é melhor**. O desvio abaixo é `numpy.std()` (`ddof=0`) e descreve a variação entre os cinco folds; não é intervalo de confiança.

In [ ]:
scores_m = cross_val_score(
    LinearRegression(), X_treino_m, y_treino,
    cv=kf, scoring="neg_mean_absolute_error"
)
mae_m = -scores_m
print("Scores retornados:", np.round(scores_m, 0))
print("MAE por parte:    ", np.round(mae_m, 0))
print(f"Regressão múltipla: US$ {mae_m.mean():,.0f} ± {mae_m.std():,.0f}")

> **Pare e explique:** os scores negativos significam “erro negativo”? O que descrevem a média e o desvio das cinco notas?

**Sua resposta:** …

## 4. Laboratório: compare nas mesmas cinco partes

**Tarefa 1.** Avalie a regressão simples (apenas `horsepower`) e a referência que sempre prevê a média do treino (`DummyRegressor`). Use **o mesmo `kf`** e a mesma métrica do exemplo. Troque o sinal para obter `mae_s` e `mae_ref`.

In [ ]:
# TODO 1 — complete as duas chamadas. Use X_treino_s, y_treino e cv=kf.
# mae_s = -cross_val_score(...)
# mae_ref = -cross_val_score(...)
# print(...)

**Tarefa 2.** Monte uma tabela com `parte`, `referencia`, `simples`, `multipla`. Depois resuma cada candidato com MAE médio e desvio. Compare **parte a parte**: a múltipla venceu quantas vezes? Registrar os cinco valores ajuda a enxergar a estabilidade da escolha.

In [ ]:
# TODO 2 — crie por_parte e resumo. Use np.std ou pandas .std(ddof=0).
# por_parte = pd.DataFrame({...}).set_index("parte")
# resumo = pd.DataFrame({...})
# print(...)
# display(por_parte, resumo)

In [ ]:
# Execute depois de completar as tarefas 1 e 2.
ax = por_parte[["simples", "multipla"]].plot(
    marker="o", figsize=(8, 4), color=["#8066C2", "#108A78"]
)
ax.set(title="MAE de validação em cada parte", xlabel="Parte", ylabel="MAE (US$)")
ax.set_xticks(por_parte.index)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

> **Registre sua interpretação:** qual modelo teve o menor MAE médio? O desvio da múltipla é maior ou menor que o da simples? A diferença entre as médias parece convincente quando você olha cada parte?

**Sua resposta:** …

## 5. Flexibilidade e pré-processamento dentro de cada fold

Vamos testar graus 1, 4 e 8 usando apenas `horsepower`. O `StandardScaler` e o `PolynomialFeatures` ficam **dentro** do `Pipeline`: em cada rodada, o escalonamento é ajustado apenas nos 128 carros de ajuste. `cross_validate(return_train_score=True)` permite comparar o MAE de ajuste e o de validação. **Não use `fit_transform()` na base inteira antes do KFold.**

**Tarefa 3.** Complete o laço com `Pipeline`, `cross_validate` e as médias dos erros de treino e validação.

In [ ]:
# TODO 3 — complete. Mantenha escala e polinômio dentro do Pipeline.
# linhas_polinomios = []
# for grau in [1, 4, 8]:
#     modelo = Pipeline([...])
#     resultado = cross_validate(
#         modelo, X_treino_s, y_treino, cv=kf,
#         scoring="neg_mean_absolute_error", return_train_score=True
#     )
#     linhas_polinomios.append({...})
# polinomios = pd.DataFrame(linhas_polinomios).set_index("grau")
# polinomios

> **Pare e explique:** no grau 8, qual MAE é menor: treino ou validação? Isso sugere subajuste ou sobreajuste? Por que o escalonamento precisa estar dentro do `Pipeline`?

**Sua resposta:** …

## 6. Decida com a validação; só depois abra o teste

Na comparação principal, considere **simples** e **múltipla**. Registre a escolha e a justificativa **antes** de executar a próxima etapa. Se houvesse empate próximo da oscilação entre folds, a opção mais simples mereceria preferência. Aqui a vantagem da múltipla aparece nas cinco partes, e suas quatro variáveis são conhecidas no momento da previsão.

**Sua decisão (antes do teste):** escolhi __________ porque __________. Cite os dois MAEs médios, os desvios e o resultado parte a parte.

In [ ]:
# TODO 4 — somente depois de registrar sua escolha acima.
# colunas_escolhidas = ...
# modelo_final = LinearRegression()
# modelo_final.fit(X_treino[colunas_escolhidas], y_treino)
# previsoes_teste = modelo_final.predict(X_teste[colunas_escolhidas])
# mae_teste = mean_absolute_error(...)
# rmse_teste = np.sqrt(mean_squared_error(...))
# r2_teste = r2_score(...)
# print(...)

> **Pare e explique:** o MAE de teste ficou acima ou abaixo do MAE médio de validação? Isso prova que a validação cruzada “falhou”? Por que não devemos mudar as variáveis agora para melhorar o resultado de teste?

**Sua resposta:** …

## 7. Investigue os erros e escreva o relatório

**Tarefa 5.** Monte uma tabela com preço observado, previsto e erro absoluto no teste. Use `price > 30000` para uma **análise descritiva posterior**, separando carros comuns e de luxo. `price` não foi usado como entrada do modelo. Conte os carros e calcule o MAE de cada grupo. Isso explica uma limitação do resultado, sem reabrir a escolha do modelo.

In [ ]:
# TODO 5 — tabela de erros e resumo por faixa.
# erros_teste = pd.DataFrame({...}, index=y_teste.index)
# erros_teste["erro_absoluto"] = ...
# erros_teste["faixa"] = np.where(...)
# por_faixa = erros_teste.groupby("faixa").agg(...)
# display(por_faixa, erros_teste.nlargest(5, "erro_absoluto"))

### Seu relatório (4–6 frases)

Escreva no formato do slide 19 da aula: **como escolheu**, **MAE médio ± desvio na validação para os candidatos**, **MAE/RMSE/R² no teste**, **limitação observada** e **próximo passo para um novo ciclo**. Use os números que o seu notebook produziu.

> **Relatório:** …

**Pergunta de saída:** se outro sorteio do teste desse MAE de US$ 1.700, o modelo já treinado seria melhor? Explique usando a ideia de “divisão como amostra”.

> **Resposta:** …

## Entrega e desafio opcional

**Entregue o notebook com os cinco `TODO` completos e suas respostas em Markdown.** Verifique: teste separado; mesmo `KFold` nos candidatos; sinal do MAE corrigido; média e desvio; pré-processamento dentro do `Pipeline`; escolha registrada antes do teste; relatório final com limitação.

**Desafio opcional:** repita a validação da múltipla com `k = 2` e `k = 10`, sempre com `shuffle=True, random_state=42`. Compare a média e o desvio. A parte de validação menor parece mais instável? Mantenha os 41 carros de teste fora desse experimento.